# 07_model_training_baselines_and_ml.ipynb

This notebook trains baseline strategies and machine learning models for setup classification and meta-labeling, incorporating feature selection and class balancing.

### Objectives:
1. Load the features and labels.
2. Split the data chronologically (e.g. Train: 60%, Val: 20%, Test: 20%).
3. Apply correlation-based feature selection to reduce redundancy.
4. Evaluate baseline strategies.
5. Train a Stage 2 meta-classifier with class balancing (predicting trade success probability).
6. Calibrate probability outputs using Platt scaling or Isotonic Regression.
7. Save the trained models and calibration objects.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Configurations, Features & Labels

In [ ]:
from utils.config import load_global_config
from utils.io_utils import load_parquet

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

# Load features
features_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_5m_features.parquet")
df_features = load_parquet(features_path)

# Load labels
labels_path = os.path.join(PROJECT_ROOT, "data", "labels", f"{symbol}_5m_labels.parquet")
df_labels = load_parquet(labels_path)

# Align them: only keep rows where we have labels (the event timestamps)
df_model = df_features.loc[df_labels.index].copy()
df_model["label"] = df_labels["label"].astype(float)
df_model["meta_label"] = df_labels["meta_label"].astype(float)

print(f"Aligned dataset size: {df_model.shape}")

## 3. Feature Selection (Correlation Filter)

We drop highly correlated features to prevent overfitting.

In [ ]:
from features.selection import remove_highly_correlated_features

exclude_cols = ["label", "meta_label", "timestamp", "open", "high", "low", "close", "volume", "taker_buy_base_volume"]
raw_features = [col for col in df_model.columns if col not in exclude_cols]

# Remove features with > 0.85 correlation
features = remove_highly_correlated_features(df_model, raw_features, threshold=0.85)
print(f"Selected {len(features)} features out of {len(raw_features)}.")

## 4. Chronological Split

In [ ]:
n_samples = len(df_model)
train_idx = int(n_samples * 0.6)
val_idx = int(n_samples * 0.8)

df_train = df_model.iloc[:train_idx]
df_val = df_model.iloc[train_idx:val_idx]
df_test = df_model.iloc[val_idx:]

print(f"Train size: {len(df_train)}, Val size: {len(df_val)}, Test size: {len(df_test)}")

## 5. Evaluate Baselines

In [ ]:
from models.baselines import BaselineModel
from sklearn.metrics import accuracy_score, classification_report

baseline = BaselineModel(strategy="trend_following")
preds = baseline.predict(df_val)

y_val_binary = (df_val["label"] == 1).astype(int)
pred_binary = (preds == 1).astype(int)

print("Trend Following Baseline Accuracy on Val Set:")
print(accuracy_score(y_val_binary, pred_binary))
print(classification_report(y_val_binary, pred_binary))

## 6. Train Machine Learning Models (LightGBM with Class Balancing)

We train a LightGBM model to predict the meta-label (whether a setup will succeed or fail), incorporating `class_weight='balanced'`.

In [ ]:
from models.tabular_models import TabularModelWrapper
from models.calibration import ProbabilityCalibrator
from models.model_registry import ModelRegistry

X_train = df_train[features].values
y_train = df_train["meta_label"].values
X_val = df_val[features].values
y_val = df_val["meta_label"].values

# Train LightGBM model with class balancing
model_wrapper = TabularModelWrapper(
    model_type="lightgbm", 
    params={
        "n_estimators": 50, 
        "learning_rate": 0.05, 
        "class_weight": "balanced",
        "random_state": 42
    }
)
model_wrapper.fit(X_train, y_train)

# Calibrate probabilities on the validation set
raw_probs_val = model_wrapper.predict_proba(X_val)
calibrator = ProbabilityCalibrator(method="isotonic")
calibrator.fit(raw_probs_val, y_val)

calibrated_probs_val = calibrator.calibrate(raw_probs_val)
print("Calibrated probabilities on Val Set (first 5 samples):")
print(calibrated_probs_val[:5])

## 7. Save Models & Calibrators

In [ ]:
registry = ModelRegistry(PROJECT_ROOT)
registry.save_model(model_wrapper, f"{symbol}_lgbm_model")
registry.save_model(calibrator, f"{symbol}_lgbm_calibrator")
print("Models and calibrators saved successfully.")

## Summary of Trained Models

We successfully trained and saved:
1. `models/trained_models/BTCUSDT_lgbm_model.joblib` (Balanced Meta-classifier)
2. `models/trained_models/BTCUSDT_lgbm_calibrator.joblib` (Isotonic probability calibrator)

**Next Step**: Run [08_walk_forward_validation.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/08_walk_forward_validation.ipynb) to validate the model's out-of-sample performance via walk-forward splits.